In [7]:
import re
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from colorama import Fore
from urllib.parse import urlparse
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import GaussianNB
from tld import get_tld, is_tld
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from app.ml.Preprocessing import URLFeatureExtractor
from app.ml.Preprocessing import URLDomainExtractor

from xgboost import XGBClassifier

In [6]:
import os
print(os.getcwd())

c:\Users\Lenovo\Documents\02 Programming\CPF Government app\project\backend


In [5]:
os.chdir("..")

In [ ]:
def _to_english(text: str):
    """
    Returns (translated_text, success_flag)

    - If text is English → returns original text, True
    - If translation succeeds → returns translated text, True
    - If translation fails → returns None, False
    """
    try:
        from langdetect import detect
        from deep_translator import GoogleTranslator

        if detect(text) == "en":
            print("[translator] Text detected as English, skipping translation.")
            return text, True

        translated = GoogleTranslator(source="auto", target="en").translate(text)
        if translated:
            print("[translator] Translation successful.")
            return translated, True
        print(["[translator] Translation failed or returned empty."])
    except Exception:
        pass

    return None, False

In [ ]:
input_file = "app/ml/datasets/archive/deepseek generated benign sms.txt"
output_file = "app/ml/datasets/archive/deepseek generated benign sms.csv"

cleaned_lines = []

# Read file with UTF-8 encoding
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        # Remove leading numbering like "1. ", "2. ", etc.
        cleaned_line = re.sub(r"^\d+\.\s*", "", line.strip())
        translated_line, success = _to_english(cleaned_line)
        # Skip empty lines
        if cleaned_line:
            cleaned_lines.append(translated_line)

# Convert to DataFrame
df = pd.DataFrame(cleaned_lines, columns=["text"])

# Save as CSV with UTF-8 encoding


print("Done! CSV saved as:", output_file)

In [15]:
df.to_csv(output_file, index=False, encoding="utf-8")

In [ ]:
# example marketing sms generated by deepseek  
df1 = pd.read_csv("app/ml/datasets/archive/deepseek generated benign sms.csv")

In [ ]:
# the sms spam collection dataset from kaggle
# https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset/data
df2 = pd.read_csv("app/ml/datasets/sms_spam_collection.csv", encoding="latin-1")

In [ ]:
# the sms phishing dataset from mendley
# https://data.mendeley.com/datasets/f45bkkt8pr/1
df3 = pd.read_csv("app/ml/datasets/sms_Phishing_Dataset.csv", encoding="utf-8")

In [13]:
df1.head()

,text
0,For the one who deserves elegance. Discover lu...
1,Elite furniture – redefining comfort coming so...
2,Celebrate the holidays responsibly. Keep your ...
3,Eid outfit from us? Definitely! Hurry to Moda ...
4,The perfect gift awaits. Stunning silver jewel...


In [14]:
df1["label"] = 0

In [16]:
df1.head()

,text,label
0,For the one who deserves elegance. Discover lu...,0
1,Elite furniture – redefining comfort coming so...,0
2,Celebrate the holidays responsibly. Keep your ...,0
3,Eid outfit from us? Definitely! Hurry to Moda ...,0
4,The perfect gift awaits. Stunning silver jewel...,0


In [17]:
df2.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [20]:
df2["text"] = df2["v2"]
df2["label"] = df2["v1"].map({"ham": 0, "spam": 1})

In [21]:
df2 = df2[["text", "label"]]

In [22]:
df2.head()

,text,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [26]:
df2.value_counts("label")

label
0    4825
1     747
Name: count, dtype: int64

In [23]:
df3.head()

,LABEL,TEXT,URL,EMAIL,PHONE
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No
1,ham,What's up? Do you want me to come online? If y...,No,No,No
2,ham,So u workin overtime nigpun?,No,No,No
3,ham,"Also sir, i sent you an email about how to log...",No,No,No
4,Smishing,Please Stay At Home. To encourage the notion o...,No,No,No


In [25]:
df3.value_counts("LABEL")

LABEL
ham         4844
Smishing     616
spam         466
Spam          23
smishing      22
Name: count, dtype: int64

In [ ]:
df3["label"] = df3["LABEL"].map({"ham": 0, "Spam": 1, "spam": 1,"Smishing": 2, "smishing": 2})
df3["text"] = df3["TEXT"]
df3 = df3[["text", "label"]]

In [31]:
df3.head()

,text,label
0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0
1,What's up? Do you want me to come online? If y...,0
2,So u workin overtime nigpun?,0
3,"Also sir, i sent you an email about how to log...",0
4,Please Stay At Home. To encourage the notion o...,2


In [32]:
df3.value_counts("label")

label
0    4844
2     638
1     489
Name: count, dtype: int64

In [33]:
df = pd.concat([df1, df2, df3], ignore_index=True)

In [34]:
df.head()

,text,label
0,For the one who deserves elegance. Discover lu...,0
1,Elite furniture – redefining comfort coming so...,0
2,Celebrate the holidays responsibly. Keep your ...,0
3,Eid outfit from us? Definitely! Hurry to Moda ...,0
4,The perfect gift awaits. Stunning silver jewel...,0


In [35]:
df["label"].value_counts()

label
0    10770
1     1236
2      638
Name: count, dtype: int64

 since our dataset is highly imbalanced, we will use another dataset to upsample the phishing examples

In [64]:
# https://github.com/wspr-ncsu/sms-phishing
df4 = pd.read_csv("app/ml/datasets/phishing_messages.csv")

In [65]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68029 entries, 0 to 68028
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   messageID           68029 non-null  object 
 1   objectID            68029 non-null  int64  
 2   destination number  68029 non-null  object 
 3   message             68029 non-null  object 
 4   time                68029 non-null  float64
 5   error in time       68029 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 3.6+ MB


In [66]:
df4.head()

,messageID,objectID,destination number,message,time,error in time
0,627001980f51cabb09ccf227,447883196637,"Join today and enjoy 400% casino bonus, START ...",VIP Club,1.651511e+09,7200.0
1,627001ca4bbc0d101565a3c8,436703061433,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,Top 4You,1.651530e+09,120.0
2,62734e2618226a2f38733d2c,447591156667,An up to £3 FREEBIE has been dropped onto your...,+447903576999,1.651455e+09,0.0
3,627001e1438a83d5e42455f8,17312148616,We just missed you today! Schedule a redeliver...,Google,1.651490e+09,0.0
4,62715d86a36b4bc273623c5b,18458695131,"In order to avoid any loss to you, please corr...",Telegram,1.651462e+09,0.0


In [67]:
df4["destination number"].head()

0    Join today and enjoy 400% casino bonus, START ...
1    Einzigartiges Angebot- 200% bis zu 100 EUR +50...
2    An up to £3 FREEBIE has been dropped onto your...
3    We just missed you today! Schedule a redeliver...
4    In order to avoid any loss to you, please corr...
Name: destination number, dtype: object

In [68]:
df4["text"] = df4["destination number"]
df4.head()

,messageID,objectID,destination number,message,time,error in time,text
0,627001980f51cabb09ccf227,447883196637,"Join today and enjoy 400% casino bonus, START ...",VIP Club,1.651511e+09,7200.0,"Join today and enjoy 400% casino bonus, START ..."
1,627001ca4bbc0d101565a3c8,436703061433,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,Top 4You,1.651530e+09,120.0,Einzigartiges Angebot- 200% bis zu 100 EUR +50...
2,62734e2618226a2f38733d2c,447591156667,An up to £3 FREEBIE has been dropped onto your...,+447903576999,1.651455e+09,0.0,An up to £3 FREEBIE has been dropped onto your...
3,627001e1438a83d5e42455f8,17312148616,We just missed you today! Schedule a redeliver...,Google,1.651490e+09,0.0,We just missed you today! Schedule a redeliver...
4,62715d86a36b4bc273623c5b,18458695131,"In order to avoid any loss to you, please corr...",Telegram,1.651462e+09,0.0,"In order to avoid any loss to you, please corr..."


In [72]:
df4["label"] = 2
df4 = df4[["text", "label"]]

In [73]:
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 68029 entries, 0 to 68028
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    68029 non-null  object
 1   label   68029 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.6+ MB


In [74]:
# count na and duplicates
print(df4.isna().sum())
print(df4.duplicated().sum())

text     0
label    0
dtype: int64
30554


In [75]:
df4["text"]

0        Join today and enjoy 400% casino bonus, START ...
1        Einzigartiges Angebot- 200% bis zu 100 EUR +50...
2        An up to £3 FREEBIE has been dropped onto your...
3        We just missed you today! Schedule a redeliver...
4        In order to avoid any loss to you, please corr...
                               ...                        
68024    New RHRY L0gin 【 www.bitoii.com 】 Username:Rom...
68025    New RHRY L0gin 【 www.bitoii.com 】 Username:Rom...
68026    sms180217@2-tdbanking.sms.com / Zelle Detected...
68027    sms370017@2-tdbanking.sms.com / Zelle Detected...
68028    \nChronopost : Votre colis est en attente d'in...
Name: text, Length: 68029, dtype: object

In [76]:
df4.drop_duplicates(inplace=True)
df4["text"].dropna(inplace=True)
df4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37475 entries, 0 to 68028
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    37475 non-null  object
 1   label   37475 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 878.3+ KB


In [77]:
# clean white space from df4["text"]
df4["text"] = df4["text"].str.strip()

In [79]:
# randomly sample 1000 rows from df4 to prevent it from dominating the dataset
df4 = df4.sample(n=2000, random_state=42, ignore_index=True)

In [78]:
df4.head(10)

,text,label
0,"Join today and enjoy 400% casino bonus, START ...",2
1,Einzigartiges Angebot- 200% bis zu 100 EUR +50...,2
2,An up to £3 FREEBIE has been dropped onto your...,2
3,We just missed you today! Schedule a redeliver...,2
4,"In order to avoid any loss to you, please corr...",2
5,"[BN] 地 被限制使用, 立即更新: binenae.com",2
6,"[BN] 地 被限制使用, 立即更新: binenae.com",2
7,An unauthorised transaction was perfor...,2
9,: Your 'CHECKING' and 'DEBlT' have been frozen...,2
14,An unauthorised transaction was performed usin...,2


In [81]:
df = pd.concat([df, df4], ignore_index=True)

In [82]:
# inspect the dataset
df.to_csv("app/ml/datasets/sms_combined.csv", index=False, encoding="utf-8")

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14644 entries, 0 to 14643
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    14644 non-null  object
 1   label   14644 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 228.9+ KB


In [84]:
df["label"].value_counts()

label
0    10770
2     2638
1     1236
Name: count, dtype: int64